# cell2location — Step 2: spatial mapping (per chunk)

This notebook is generated from the cell2location `spatial-mapping` skill.
See [../SKILL.md](../SKILL.md) for the full workflow.

It is based on the `cell2state_embryo` papermill notebook (the only published
notebook with correct settings for stratified per-sample chunking and the
nuclei-count model), simplified for general use.

The notebook runs ONE training chunk. For `n_chunks > 1`, submit one instance
per `training_batch` index (0..n_chunks-1) using the launchers in this
directory ([bsub.sh](bsub.sh) for LSF, [sbatch.sh](sbatch.sh) for Slurm,
[run_local.sh](run_local.sh) for local GPU), then aggregate chunk outputs
with [step2_aggregate_chunks.ipynb](step2_aggregate_chunks.ipynb).

See [../reference/hyperparameters_extract.md §1.2](../reference/hyperparameters_extract.md)
and [../reference/fig_S27_hyperparameters.png](../reference/fig_S27_hyperparameters.png)
for hyperparameter defaults and decision rules.


In [ ]:
# === PARAMETERS (papermill) ===
# Set via:
#   papermill step2_spatial_mapping.ipynb out.ipynb -p spatial_h5ad_path ... -p signatures_csv ...
# The spatial-mapping skill walks the user through each value. Reference:
# - decision tree:  ../reference/fig_S27_hyperparameters.png
# - defaults:       ../reference/hyperparameters_extract.md

# ---- I/O ----
spatial_h5ad_path = ""        # spatial AnnData (Visium / Visium-HD / Cytassist / Slide-seq / Stereo-seq)
signatures_csv = ""           # step1 output (genes x cell_types, linear scale, batch-corrected)
output_dir = "./spatial_mapping_output"
output_name = "c2l_run"

# ---- Chunking ----
training_batch = 0            # 0..n_chunks-1; one notebook execution per chunk
n_chunks = 1                  # 1 = no chunking (full-batch on one GPU); skill computes per Phase 5
seed = 0

# ---- Training ----
max_epochs = 30000            # medium-tier default; see SKILL.md Phase 8

# ---- Detection sensitivity (Phase 4 / Fig S27 lower flow) ----
detection_alpha = None        # MUST be set: 20 (high variability) or 200 (low)

# ---- N_cells_per_location (Phase 3 / Fig S27 upper flow) ----
# Set EXACTLY ONE of the column / scalar options.
N_cells_per_location_column = None   # column in adata.obs with per-location N_s
                                     #   (e.g. "n_cell_occupancy" = occupancy * N_nuclei * scaling).
                                     #   REQUIRES installing cell2location from the
                                     #   hires_sliding_window branch (see SKILL.md Phase 6).
N_cells_per_location_scalar = None   # tissue-level scalar from Fig S27 manual count
                                     #   or cell-size fallback (Visium=5, Slide-seq V2=1, ...).
N_cells_per_location_alpha_prior = None  # v^n: 1 (global N) / 1000 (per-location N_s + hires)

# ---- Branch features (only effective with hires_sliding_window installed) ----
use_proportion_factorisation_prior_on_w_sf = False  # True with per-location N_s
use_n_s_cells_per_location_limit = False            # True with per-location N_s
A_B_per_location_alpha_prior = None
use_per_cell_type_normalisation = False             # advanced

# ---- Factorisation (supplement §1.2) ----
n_groups = 50
A_factors_per_location = 7.0
B_groups_per_location = 7.0   # NB: embryo used 5; supplement default is 7

# ---- QC + gene filtering ----
total_counts_min = 1000
total_counts_max = 200000
sample_fraction_threshold = 0.7
gene_filter_cell_count_cutoff = 15
gene_filter_cell_percentage_cutoff2 = 0.15
gene_filter_nonz_mean_cutoff = 1.11

# ---- Posterior export ----
use_quantiles = True          # MANDATORY for n_obs > 100k; see issue #278
compute_expected = False      # set True if you need per-cell-type per-gene expression layers


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import scanpy as sc
import anndata
import scvi
import cell2location
from cell2location.utils.filtering import filter_genes
import matplotlib.pyplot as plt
import pyro

scvi.settings.seed = seed
np.random.seed(seed)


## Load spatial data + step 1 signatures

In [ ]:
# Load spatial AnnData and step1 reference signatures
adata_vis = sc.read_h5ad(spatial_h5ad_path)
inf_aver = pd.read_csv(signatures_csv, index_col=0)
print(f"Spatial:     n_obs={adata_vis.n_obs}, n_vars={adata_vis.n_vars}")
print(f"Signatures:  {inf_aver.shape[0]} genes x {inf_aver.shape[1]} cell types")
print(f"Samples:     {adata_vis.obs['sample'].nunique() if 'sample' in adata_vis.obs.columns else 'no sample column'}")


## QC filtering (per-spot + gene filtering)

In [ ]:
# Per-spot QC filtering (embryo workflow QC, parameterised)
# - drop locations outside [total_counts_min, total_counts_max]
# - drop samples where <sample_fraction_threshold of locations pass QC
ind = (
    (adata_vis.obs['total_counts'] > total_counts_min).values
    & (adata_vis.obs['total_counts'] < total_counts_max).values
)

# If user has nuclei segmentation column, also require at least one nucleus.
if N_cells_per_location_column is not None and N_cells_per_location_column in adata_vis.obs.columns:
    ind = ind & (adata_vis.obs[N_cells_per_location_column] > 0).values

# Sample-fraction filter: drop samples where <70% of locations pass
fraction_selected = (
    adata_vis.obs['sample'][ind].value_counts()
    / adata_vis.obs['sample'].value_counts()
).sort_values()
ind = ind & adata_vis.obs['sample'].isin(
    fraction_selected[fraction_selected > sample_fraction_threshold].index
).values
adata_vis = adata_vis[ind, :].copy()
print(f"After QC: {adata_vis.n_obs} locations, {adata_vis.obs['sample'].nunique()} samples")

# Gene filter (cell2location.utils.filter_genes)
selected = filter_genes(
    adata_vis,
    cell_count_cutoff=gene_filter_cell_count_cutoff,
    cell_percentage_cutoff2=gene_filter_cell_percentage_cutoff2,
    nonz_mean_cutoff=gene_filter_nonz_mean_cutoff,
)
# NB: embryo workflow computed filter but did NOT subset. Match that behaviour;
# the cell2location model handles non-selected genes via background prior.
# Uncomment the next line to subset.
# adata_vis = adata_vis[:, selected].copy()

# Find shared genes with the reference signatures
intersect = np.intersect1d(adata_vis.var_names, inf_aver.index)
adata_vis = adata_vis[:, intersect].copy()
inf_aver = inf_aver.loc[intersect, :].copy()
print(f"After intersecting with signatures: {adata_vis.n_vars} genes")


## Stratified chunk assignment

In [ ]:
# Stratified chunk assignment (Phase 5):
# Each chunk gets a random sample of locations from EVERY sample, so all samples
# are present in every training chunk (joint multi-sample modelling preserved).
if n_chunks > 1:
    if 'training_batch' not in adata_vis.obs.columns:
        adata_vis.obs['training_batch'] = 0
        for sample in adata_vis.obs['sample'].unique():
            mask = adata_vis.obs['sample'] == sample
            adata_vis.obs.loc[mask, 'training_batch'] = np.random.choice(
                list(range(n_chunks)), size=int(mask.sum()), replace=True
            )
    ind = adata_vis.obs['training_batch'] == training_batch
    adata_vis = adata_vis[ind, :].copy()
    print(f"Chunk {training_batch}/{n_chunks - 1}: {adata_vis.n_obs} locations")


## Resolve N_cells_per_location + sanity-check detection_alpha

In [ ]:
# Resolve N_cells_per_location:
# - If a column was provided, use per-location array (and the hires-branch flags MUST be set).
# - Else use the scalar (Fig S27 fallback).
# - If both None, fail loudly.
if N_cells_per_location_column is not None:
    if N_cells_per_location_column not in adata_vis.obs.columns:
        raise ValueError(
            f"N_cells_per_location_column={N_cells_per_location_column!r} not in adata_vis.obs. "
            f"Available: {list(adata_vis.obs.columns)}"
        )
    N_cells_per_location = adata_vis.obs[[N_cells_per_location_column]].values.astype('float32')
    # Embryo formula reminder: n_cell_occupancy = occupancy * N_nuclei * scaling.
    # Effectiveness REQUIRES installing cell2location from hires_sliding_window
    # branch AND setting use_proportion_factorisation_prior_on_w_sf=True AND
    # use_n_s_cells_per_location_limit=True.
elif N_cells_per_location_scalar is not None:
    N_cells_per_location = float(N_cells_per_location_scalar)
else:
    raise ValueError(
        "Set either N_cells_per_location_column (per-location array) or "
        "N_cells_per_location_scalar (tissue-level scalar; e.g. 5 for Visium, "
        "1 for Slide-seq V2). See Fig S27 in ../reference/."
    )

# Sanity check on detection_alpha
if detection_alpha is None:
    raise ValueError(
        "Set detection_alpha to 20 (high within-batch variability — FFPE / Cytassist / "
        "Visium-HD / older human samples) or 200 (low variability — fresh-frozen single-sample Visium). "
        "See Fig S27 in ../reference/."
    )


## Auto-derived hyperpriors

In [ ]:
# Auto-derived hyperpriors (supplement §1.2 item 3 + embryo cells 36-39).
# These are computed from data; user does not set them directly.
if isinstance(N_cells_per_location, np.ndarray):
    N_mean = float(np.mean(N_cells_per_location))
else:
    N_mean = float(N_cells_per_location)

expected_y_e = (
    adata_vis.obs[['sample', 'total_counts']].groupby('sample').mean()
    / (inf_aver.sum(0) * N_mean).mean()
)
mean_alpha_prior = float(np.round(
    ((expected_y_e.mean() ** 2) / expected_y_e.var()).values[0] / 3, 2
))
detection_cell_type_prior_alpha = float(np.round(
    ((inf_aver.sum(0).mean() ** 2) / inf_aver.sum(0).var()) * 20, 2
))
print(f"Auto-derived: mean_alpha_prior={mean_alpha_prior}, "
      f"detection_cell_type_prior_alpha={detection_cell_type_prior_alpha}")


## Setup + train Cell2location

In [ ]:
# Setup + train Cell2location (per supplement §1.3 + embryo cell 44).
# REFUSALS (enforced by skill, surfaced as comments here):
#   - batch_size != None         (mini-batch is REFUSED for spatial mapping)
#   - log-transformed input       (REFUSED; cell2location needs linear counts)
#   - num_samples=1000 on n_obs > 100k without use_quantiles=True (REFUSED, OOM)
cell2location.models.Cell2location.setup_anndata(adata=adata_vis, batch_key="sample")

# Build kwargs dynamically because some are only valid on hires_sliding_window.
model_kwargs = dict(
    cell_state_df=inf_aver,
    amortised=False,
    N_cells_per_location=N_cells_per_location,
    detection_alpha=float(detection_alpha),
    detection_hyp_prior={"mean_alpha": mean_alpha_prior},
    A_factors_per_location=A_factors_per_location,
    B_groups_per_location=B_groups_per_location,
    n_groups=n_groups,
)
# hires-branch-only kwargs
if use_proportion_factorisation_prior_on_w_sf or use_n_s_cells_per_location_limit:
    model_kwargs.update(dict(
        use_proportion_factorisation_prior_on_w_sf=use_proportion_factorisation_prior_on_w_sf,
        use_n_s_cells_per_location_limit=use_n_s_cells_per_location_limit,
        N_cells_per_location_alpha_prior=N_cells_per_location_alpha_prior,
        A_B_per_location_alpha_prior=A_B_per_location_alpha_prior,
        use_per_cell_type_normalisation=use_per_cell_type_normalisation,
        detection_cell_type_prior_alpha=detection_cell_type_prior_alpha,
    ))

mod = cell2location.models.Cell2location(adata_vis, **model_kwargs)
mod.view_anndata_setup()

mod.train(
    max_epochs=max_epochs,
    batch_size=None,                 # FULL BATCH — DO NOT CHANGE (supplement §1.3, issue #356)
    plan_kwargs={'optim': pyro.optim.Adam(optim_args={'lr': 0.002})},
    train_size=1,
    scale_elbo=1 / (adata_vis.n_obs * adata_vis.n_vars),
    accelerator='gpu',
)

run_dir = os.path.join(output_dir, f"{output_name}_chunk{training_batch}")
os.makedirs(run_dir, exist_ok=True)
mod.save(run_dir, overwrite=True)


## Posterior export

In [ ]:
# Posterior export (supplement §1.3 + issues #278/#360)
adata_vis = mod.export_posterior(
    adata_vis,
    sample_kwargs={
        'batch_size': int(np.ceil(adata_vis.n_obs / 4)),
        'accelerator': 'gpu',
        'return_observed': False,
    },
    add_to_obsm=['means', 'q05', 'q50', 'q95'],
    use_quantiles=use_quantiles,
    exclude_vars=['data_target'],
)

if compute_expected:
    # Optional per-cell-type expression layers (memory-intensive; per-chunk only)
    expected_dict = mod.module.model.compute_expected_per_cell_type(
        mod.samples["post_sample_q05"], mod.adata_manager
    )
    for i, n in enumerate(mod.factor_names_):
        adata_vis.layers[n] = expected_dict['mu'][i]

# Save chunk results
adata_file = os.path.join(run_dir, "sp.h5ad")
adata_vis.write(adata_file)
print(f"Wrote chunk output: {adata_file}")


## QC plots

In [ ]:
# QC plots (supplement §1.3 recommendations)
# 1. ELBO history — should plateau; late oscillations are normal (issue #327)
mod.plot_history(5000)
plt.savefig(os.path.join(run_dir, "training_ELBO_history_minus5k.png"), bbox_inches='tight')
plt.show()
plt.close()
mod.plot_history(0)
plt.savefig(os.path.join(run_dir, "training_ELBO_history_all.png"), bbox_inches='tight')
plt.show()
plt.close()

# 2. Posterior predictive check — log10(mu+1) vs log10(d+1) should be near-diagonal
mod.plot_QC(summary_name='q05')
plt.savefig(os.path.join(run_dir, "reconstruction_accuracy_histogram.png"), bbox_inches='tight')
plt.show()
plt.close()
